# 03 - Identify Face-Relevant Units (fc8)

Finds which specific AlexNet fc8 units matter most for predicting FFA
activity, using RidgeCV regression weight magnitude.

**Updated:** now targets fc8 instead of fc7, since the validated
baseline (with RidgeCV, standardization, per-voxel alpha selection)
confirmed fc8 is FFA's best-predicting layer (accuracy 0.2657),
not fc7 as originally assumed before validation.

This is the step the ablation hypothesis depends on:

> Can targeted removal of DNN features that are important for
> predicting activity in face-selective cortex selectively impair
> prediction of fMRI responses in the FFA, while leaving predictions
> for other visual regions relatively intact?

In [ ]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

In [ ]:
!pip install nilearn decord python-dotenv --quiet

In [ ]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")

In [ ]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

from src.data_loading import (
    download_algonauts_data,
    load_all_rois,
    get_video_list,
    load_video_frames,
)
from src.models.alexnet_wrapper import AlexNetWrapper

In [ ]:
download_algonauts_data(data_dir="data/raw")

fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"
roi_data = load_all_rois(fmri_dir, subject)

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]
print("FFA shape:", roi_data["FFA"].shape)

## Load or extract fc8 activations

Reuses the cached AlexNet activations if available, extracting from
scratch otherwise.

In [ ]:
os.makedirs("data/processed", exist_ok=True)
activation_cache_path = "data/processed/alexnet_activations.npz"

need_to_extract = True

if os.path.exists(activation_cache_path):
    cached = np.load(activation_cache_path, allow_pickle=True)
    all_activations = cached["all_activations"].item()
    cached_num_videos = next(iter(all_activations.values())).shape[0]

    if cached_num_videos == NUM_TRAIN_VIDEOS:
        print("Loaded cached activations.")
        need_to_extract = False
    else:
        print("Cache is stale, re-extracting.")
else:
    print("No cache found, extracting from scratch.")

if need_to_extract:
    video_dir = "data/raw/AlgonautsVideos268_All_30fpsmax"
    video_paths = get_video_list(video_dir)[:NUM_TRAIN_VIDEOS]

    assert len(video_paths) == NUM_TRAIN_VIDEOS, (
        f"Video count ({len(video_paths)}) does not match fMRI row count "
        f"({NUM_TRAIN_VIDEOS})."
    )

    alexnet = AlexNetWrapper()
    all_activations = {layer: [] for layer in alexnet.layers}

    for video_path in tqdm(video_paths):
        frames = load_video_frames(video_path, num_frames=8)
        layer_acts = alexnet.extract(frames)
        for layer in alexnet.layers:
            all_activations[layer].append(layer_acts[layer])

    for layer in all_activations:
        all_activations[layer] = np.stack(all_activations[layer])

    np.savez(activation_cache_path, all_activations=all_activations)
    print("Extraction complete and cached.")

X_fc8 = all_activations["fc8"]
assert X_fc8.shape[0] == NUM_TRAIN_VIDEOS, "fc8 activations don't match fMRI row count."
print("fc8 shape:", X_fc8.shape)

## Fit the FFA regression on fc8 and rank units by importance

Uses RidgeCV with alpha_per_target=True (same as the validated
production encoding model), fit on standardized features. For each
fc8 unit, we compute the mean absolute regression weight across all
FFA voxels. Units with the largest mean absolute weight are the ones
the model relies on most heavily to predict FFA activity.

In [ ]:
Y_ffa = roi_data["FFA"]

scaler = StandardScaler()
X_fc8_scaled = scaler.fit_transform(X_fc8)

alphas = np.logspace(0, 6, 13)
ridge_model = RidgeCV(alphas=alphas, alpha_per_target=True)
ridge_model.fit(X_fc8_scaled, Y_ffa)

# coef_ shape: (num_voxels, num_fc8_units)
coef = ridge_model.coef_
print("Coefficient matrix shape:", coef.shape)

unit_importance = np.mean(np.abs(coef), axis=0)
print("Unit importance vector shape:", unit_importance.shape)

In [ ]:
K = 50

top_k_units = np.argsort(unit_importance)[::-1][:K]

print(f"Top {K} fc8 units most important for FFA prediction:")
print(top_k_units)

importance_df = pd.DataFrame({
    "unit_index": np.arange(len(unit_importance)),
    "importance": unit_importance,
})
importance_df = importance_df.sort_values("importance", ascending=False)

os.makedirs("results/tables/alexnet", exist_ok=True)
importance_df.to_csv("results/tables/alexnet/fc8_ffa_unit_importance.csv", index=False)

np.save("data/processed/fc8_top_k_units.npy", top_k_units)

importance_df.head(10)

## Matched random control set

Same size (K), but random units instead of the top-importance ones.
Used later as the control ablation condition.

In [ ]:
rng = np.random.default_rng(seed=42)

all_unit_indices = np.arange(len(unit_importance))
remaining_units = np.setdiff1d(all_unit_indices, top_k_units)
random_k_units = rng.choice(remaining_units, size=K, replace=False)

print(f"Random control set of {K} units (excluding top-importance units):")
print(random_k_units)

np.save("data/processed/fc8_random_k_units.npy", random_k_units)

## Download results

In [ ]:
from google.colab import files

files.download("results/tables/alexnet/fc8_ffa_unit_importance.csv")
files.download("data/processed/fc8_top_k_units.npy")
files.download("data/processed/fc8_random_k_units.npy")

Move these three files to:

- `results/tables/alexnet/fc8_ffa_unit_importance.csv`
- `data/processed/fc8_top_k_units.npy`
- `data/processed/fc8_random_k_units.npy`

then commit and push. The two `.npy` files are direct inputs to the
ablation notebook.